# COMP8851 — Canonical Dataset Registry

## Six-Dataset Benchmark Foundation

This notebook prepares and verifies the six canonical datasets used in the
COMP8851 graph-fraud benchmark:

1. YelpChi
2. Amazon
3. T-Finance
4. T-Social
5. Elliptic
6. FDCompCN

### Purpose

This notebook is model-independent.

It records:

- canonical dataset source
- mounted Kaggle path
- file size
- checksum
- dataset format
- processed-data identity
- benchmark metadata
- shared split and evaluator inputs

Raw datasets are not modified in place.

### Benchmark hardware

The controlled benchmark uses an NVIDIA T4.
When Kaggle provides T4 x2, only GPU 0 is exposed to benchmark code.

### Important

Author reproduction and unified benchmarking remain separate.
This notebook prepares the shared dataset foundation used by unified runs.

## Step 2 — Hardware Audit

This cell verifies the Kaggle runtime before any dataset processing.

Required controlled-benchmark hardware:

- NVIDIA T4
- GPU 0 only

The cell prints progress while it runs and uses a timeout so that
`nvidia-smi` cannot silently hang forever.

In [1]:
print("Python kernel is running", flush=True)

Python kernel is running


In [2]:
# ============================================================
# STEP 2 — HARDWARE AUDIT
# ============================================================

import os
import sys
import time
import platform
import subprocess

start = time.perf_counter()

def log(msg):
    elapsed = time.perf_counter() - start
    print(f"[{elapsed:6.2f}s] {msg}", flush=True)

log("START — hardware audit")

# ------------------------------------------------------------
# 1. Restrict benchmark-visible GPU to GPU 0
# ------------------------------------------------------------

log("Setting CUDA_VISIBLE_DEVICES=0")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"], flush=True)

# ------------------------------------------------------------
# 2. Basic runtime
# ------------------------------------------------------------

log("Reading Python and OS information")

print("\n===== RUNTIME =====", flush=True)
print("Python:", sys.version.replace("\n", " "), flush=True)
print("OS:", platform.platform(), flush=True)
print("Machine:", platform.machine(), flush=True)

# ------------------------------------------------------------
# 3. NVIDIA hardware
# ------------------------------------------------------------

log("Checking NVIDIA GPUs with nvidia-smi -L")

try:
    result = subprocess.run(
        ["nvidia-smi", "-L"],
        capture_output=True,
        text=True,
        timeout=20
    )

    print("\n===== NVIDIA GPU LIST =====", flush=True)
    print(result.stdout, flush=True)

    if result.stderr:
        print("STDERR:", result.stderr, flush=True)

except subprocess.TimeoutExpired:
    print("ERROR: nvidia-smi timed out after 20 seconds.", flush=True)
    raise

log("nvidia-smi check complete")

# ------------------------------------------------------------
# 4. PyTorch GPU visibility
# ------------------------------------------------------------

log("Importing PyTorch")

import torch

print("\n===== PYTORCH / CUDA =====", flush=True)
print("PyTorch:", torch.__version__, flush=True)
print("CUDA available:", torch.cuda.is_available(), flush=True)

if torch.cuda.is_available():

    print("Visible CUDA devices:", torch.cuda.device_count(), flush=True)

    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)

    print("GPU 0:", gpu_name, flush=True)
    print(
        "GPU 0 VRAM:",
        f"{props.total_memory / (1024**3):.2f} GiB",
        flush=True
    )

    if "T4" in gpu_name.upper():
        print("\nPASS: GPU 0 is NVIDIA T4.", flush=True)
    else:
        print(
            "\nWARNING: GPU 0 is not reported as NVIDIA T4.",
            flush=True
        )

else:
    print("\nFAIL: CUDA is not available.", flush=True)

log("DONE — hardware audit")

[  0.00s] START — hardware audit
[  0.00s] Setting CUDA_VISIBLE_DEVICES=0
CUDA_VISIBLE_DEVICES = 0
[  0.00s] Reading Python and OS information

===== RUNTIME =====
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
OS: Linux-6.12.90+-x86_64-with-glibc2.35
Machine: x86_64
[  0.01s] Checking NVIDIA GPUs with nvidia-smi -L

===== NVIDIA GPU LIST =====
GPU 0: Tesla T4 (UUID: GPU-e3c0b863-7d11-0d74-5e16-778c8f8ec634)
GPU 1: Tesla T4 (UUID: GPU-5ba81717-c196-7362-577e-a3fffeae6357)

[  0.05s] nvidia-smi check complete
[  0.05s] Importing PyTorch

===== PYTORCH / CUDA =====
PyTorch: 2.10.0+cu128
CUDA available: True
Visible CUDA devices: 1
GPU 0: Tesla T4
GPU 0 VRAM: 14.56 GiB

PASS: GPU 0 is NVIDIA T4.
[  9.01s] DONE — hardware audit


In [4]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

## Step 3 — Canonical Dataset Input Inventory

This step verifies that the six canonical benchmark datasets are attached
to the Kaggle notebook and records their exact mounted paths.

Datasets:

1. YelpChi
2. Amazon
3. T-Finance
4. T-Social
5. Elliptic
6. FDCompCN

This step does not load graph data into RAM or GPU.

It only checks:

- mounted Kaggle input folders
- filenames
- file sizes
- exact source paths

These paths will be frozen into the canonical dataset registry in the next step.

In [5]:
# ============================================================
# STEP 3A — TOP-LEVEL KAGGLE INPUTS
# ============================================================

from pathlib import Path
import time

start = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - start
    print(f"[{elapsed:6.2f}s] {message}", flush=True)

INPUT_ROOT = Path("/kaggle/input")

log("START — checking Kaggle input folders")

if not INPUT_ROOT.exists():
    raise FileNotFoundError("/kaggle/input does not exist.")

folders = sorted(
    [p for p in INPUT_ROOT.iterdir() if p.is_dir()],
    key=lambda p: p.name.lower()
)

print("\n===== ATTACHED KAGGLE INPUTS =====", flush=True)

if len(folders) == 0:
    print("No Kaggle inputs are attached.", flush=True)

for i, folder in enumerate(folders, start=1):
    print(f"{i:>2}. {folder}", flush=True)

print(f"\nTotal input folders: {len(folders)}", flush=True)

log("DONE — top-level input check")

[  0.00s] START — checking Kaggle input folders

===== ATTACHED KAGGLE INPUTS =====
 1. /kaggle/input/datasets

Total input folders: 1
[  0.01s] DONE — top-level input check


### Step 3B — File-Level Inventory

This cell scans the attached Kaggle dataset folders and reports every file
with its exact mounted path and size.

Important:

- dataset contents are not loaded
- large graph files are not copied
- no checksum is calculated yet
- T-Social is not loaded into memory

This is only a filesystem inventory.

In [6]:
# ============================================================
# STEP 3B — FILE INVENTORY WITH LIVE PROGRESS
# ============================================================

from pathlib import Path
import os
import time

start = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - start
    print(f"[{elapsed:6.2f}s] {message}", flush=True)

INPUT_ROOT = Path("/kaggle/input")

log("START — recursive Kaggle input scan")

records = []

folder_count = 0
file_count = 0

for root, dirs, filenames in os.walk(INPUT_ROOT):

    folder_count += 1

    log(f"Scanning folder {folder_count}: {root}")

    for filename in filenames:

        path = Path(root) / filename
        file_count += 1

        try:
            size = path.stat().st_size
        except Exception as e:
            size = -1

        records.append(
            {
                "path": str(path),
                "size_bytes": size
            }
        )

        if size >= 0:
            print(
                f"  FOUND {file_count:>3}: "
                f"{size:>15,} bytes | {path}",
                flush=True
            )
        else:
            print(
                f"  FOUND {file_count:>3}: "
                f"{'UNKNOWN':>15} | {path}",
                flush=True
            )

log(
    f"Scan complete — "
    f"{folder_count} folders, {file_count} files"
)

records.sort(key=lambda x: x["path"].lower())

print("\n" + "=" * 110, flush=True)
print("FINAL DATASET INPUT INVENTORY", flush=True)
print("=" * 110, flush=True)

for i, record in enumerate(records, start=1):

    path = record["path"]
    size = record["size_bytes"]

    if size >= 0:
        print(
            f"{i:>3}. {size:>15,} bytes | {path}",
            flush=True
        )
    else:
        print(
            f"{i:>3}. {'UNKNOWN':>15} | {path}",
            flush=True
        )

log("DONE — input inventory complete")

[  0.00s] START — recursive Kaggle input scan
[  0.00s] Scanning folder 1: /kaggle/input
[  0.00s] Scanning folder 2: /kaggle/input/datasets
[  0.00s] Scanning folder 3: /kaggle/input/datasets/pathikahmed0007
[  0.03s] Scanning folder 4: /kaggle/input/datasets/pathikahmed0007/fdcompcn
  FOUND   1:       1,859,985 bytes | /kaggle/input/datasets/pathikahmed0007/fdcompcn/comp.dgl
[  0.04s] Scanning folder 5: /kaggle/input/datasets/pathikahmed0007/amazon
  FOUND   2:     222,634,416 bytes | /kaggle/input/datasets/pathikahmed0007/amazon/Amazon.mat
[  0.06s] Scanning folder 6: /kaggle/input/datasets/pathikahmed0007/elliptic
  FOUND   3:     689,683,771 bytes | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_features.csv
  FOUND   4:       3,305,144 bytes | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_classes.csv
  FOUND   5:       4,470,584 bytes | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_edgelist.csv
[  0.07s] Scanning folder 7: /kaggle/input

### Step 3C — Six-Dataset Source Detection

This cell searches the mounted files for the expected canonical source names.

Expected datasets:

- YelpChi
- Amazon
- T-Finance
- T-Social
- Elliptic
- FDCompCN

A `FOUND` result only confirms the mounted source path.

Dataset identity will be verified more strictly in Step 4 using checksums
and dataset-level statistics.

In [7]:
# ============================================================
# STEP 3C — SIX-DATASET SOURCE DETECTION
# ============================================================

from pathlib import Path
import time

start = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - start
    print(f"[{elapsed:6.2f}s] {message}", flush=True)

INPUT_ROOT = Path("/kaggle/input")

log("START — searching for canonical dataset sources")

all_files = [
    p for p in INPUT_ROOT.rglob("*")
    if p.is_file()
]

log(f"Indexed {len(all_files)} mounted files")

patterns = {

    "YelpChi": [
        "fraudyelp",
        "yelpchi"
    ],

    "Amazon": [
        "fraudamazon",
        "amazon.mat"
    ],

    "T-Finance": [
        "tfinance"
    ],

    "T-Social": [
        "tsocial"
    ],

    "Elliptic": [
        "elliptic_txs_features",
        "elliptic_txs_classes",
        "elliptic_txs_edgelist"
    ],

    "FDCompCN": [
        "fdcompcn"
    ]
}

print("\n===== SIX-DATASET SOURCE DETECTION =====", flush=True)

summary = {}

for dataset, keywords in patterns.items():

    log(f"Checking {dataset}")

    matches = []

    for path in all_files:

        text = str(path).lower()

        if any(keyword in text for keyword in keywords):
            matches.append(path)

    summary[dataset] = matches

    print(f"\n[{dataset}]", flush=True)

    if matches:

        for path in matches:

            size = path.stat().st_size

            print(
                f"  FOUND | "
                f"{size:,} bytes | "
                f"{path}",
                flush=True
            )

    else:
        print("  NOT FOUND", flush=True)

print("\n===== DETECTION SUMMARY =====", flush=True)

for dataset, matches in summary.items():

    status = "FOUND" if matches else "NOT FOUND"

    print(
        f"{dataset:<12} : "
        f"{status} "
        f"({len(matches)} matching file(s))",
        flush=True
    )

log("DONE — six-dataset detection complete")

[  0.00s] START — searching for canonical dataset sources
[  0.03s] Indexed 8 mounted files

===== SIX-DATASET SOURCE DETECTION =====
[  0.04s] Checking YelpChi

[YelpChi]
  FOUND | 207,676,376 bytes | /kaggle/input/datasets/pathikahmed0007/yelp-chi/YelpChi.mat
[  0.04s] Checking Amazon

[Amazon]
  FOUND | 222,634,416 bytes | /kaggle/input/datasets/pathikahmed0007/amazon/Amazon.mat
[  0.04s] Checking T-Finance

[T-Finance]
  FOUND | 682,904,644 bytes | /kaggle/input/datasets/pathikahmed0007/tfinance/tfinance
[  0.04s] Checking T-Social

[T-Social]
  FOUND | 4,110,300,257 bytes | /kaggle/input/datasets/pathikahmed0007/tsocial/tsocial
[  0.04s] Checking Elliptic

[Elliptic]
  FOUND | 689,683,771 bytes | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_features.csv
  FOUND | 3,305,144 bytes | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_classes.csv
  FOUND | 4,470,584 bytes | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_edgelist.csv
[  0.05s] Ch

## Step 4 — Canonical Dataset Registry and Checksums

All six benchmark datasets were successfully detected.

This step freezes their exact Kaggle-mounted source paths and calculates
SHA-256 checksums before any model-specific processing begins.

The checksum record provides dataset-version identity for later run manifests.

### Datasets

- YelpChi
- Amazon
- T-Finance
- T-Social
- Elliptic
- FDCompCN

### Important

- Source files remain read-only under `/kaggle/input`.
- No graph transformation occurs in this step.
- Large files are hashed in chunks.
- Progress is printed while hashing.
- Elliptic contains three canonical source files, so each file receives its
  own SHA-256 value.

In [9]:
# ============================================================
# STEP 4A — FREEZE CANONICAL DATASET PATHS
# ============================================================

from pathlib import Path
import json
import time

start = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - start
    print(f"[{elapsed:7.2f}s] {message}", flush=True)


log("START — building canonical path registry")

DATASET_FILES = {

    "yelpchi": [
        Path(
            "/kaggle/input/datasets/pathikahmed0007/"
            "yelp-chi/YelpChi.mat"
        )
    ],

    "amazon": [
        Path(
            "/kaggle/input/datasets/pathikahmed0007/"
            "amazon/Amazon.mat"
        )
    ],

    "tfinance": [
        Path(
            "/kaggle/input/datasets/pathikahmed0007/"
            "tfinance/tfinance"
        )
    ],

    "tsocial": [
        Path(
            "/kaggle/input/datasets/pathikahmed0007/"
            "tsocial/tsocial"
        )
    ],

    "elliptic": [
        Path(
            "/kaggle/input/datasets/pathikahmed0007/"
            "elliptic/elliptic_txs_features.csv"
        ),
        Path(
            "/kaggle/input/datasets/pathikahmed0007/"
            "elliptic/elliptic_txs_classes.csv"
        ),
        Path(
            "/kaggle/input/datasets/pathikahmed0007/"
            "elliptic/elliptic_txs_edgelist.csv"
        ),
    ],

    "fdcompcn": [
        Path(
            "/kaggle/input/datasets/pathikahmed0007/"
            "fdcompcn/comp.dgl"
        )
    ]
}


print("\n===== CANONICAL DATASET PATH CHECK =====", flush=True)

all_ok = True

for dataset, paths in DATASET_FILES.items():

    print(f"\n[{dataset.upper()}]", flush=True)

    for path in paths:

        exists = path.exists()

        if exists:
            size = path.stat().st_size

            print(
                f"  PASS | "
                f"{size:>15,} bytes | {path}",
                flush=True
            )

        else:
            all_ok = False

            print(
                f"  FAIL | FILE NOT FOUND | {path}",
                flush=True
            )


print("\n========================================", flush=True)

if all_ok:
    print("PASS: All canonical dataset files exist.", flush=True)
else:
    raise FileNotFoundError(
        "One or more canonical dataset files are missing."
    )

log("DONE — path registry frozen")

[   0.00s] START — building canonical path registry

===== CANONICAL DATASET PATH CHECK =====

[YELPCHI]
  PASS |     207,676,376 bytes | /kaggle/input/datasets/pathikahmed0007/yelp-chi/YelpChi.mat

[AMAZON]
  PASS |     222,634,416 bytes | /kaggle/input/datasets/pathikahmed0007/amazon/Amazon.mat

[TFINANCE]
  PASS |     682,904,644 bytes | /kaggle/input/datasets/pathikahmed0007/tfinance/tfinance

[TSOCIAL]
  PASS |   4,110,300,257 bytes | /kaggle/input/datasets/pathikahmed0007/tsocial/tsocial

[ELLIPTIC]
  PASS |     689,683,771 bytes | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_features.csv
  PASS |       3,305,144 bytes | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_classes.csv
  PASS |       4,470,584 bytes | /kaggle/input/datasets/pathikahmed0007/elliptic/elliptic_txs_edgelist.csv

[FDCOMPCN]
  PASS |       1,859,985 bytes | /kaggle/input/datasets/pathikahmed0007/fdcompcn/comp.dgl

PASS: All canonical dataset files exist.
[   0.03s] DONE — path re

### Step 4B — SHA-256 Dataset Identity

Each canonical source file is hashed using SHA-256.

Files are processed sequentially so that large-dataset I/O does not compete
for memory or storage bandwidth.

For large files, progress is printed approximately every 256 MiB.

The source files are read only; they are not copied or modified.

In [10]:
# ============================================================
# STEP 4B — SHA-256 WITH LIVE PROGRESS
# ============================================================

import hashlib
import time
from pathlib import Path


GLOBAL_START = time.perf_counter()

# Read 8 MiB at a time.
CHUNK_SIZE = 8 * 1024 * 1024

# Print progress approximately every 256 MiB.
REPORT_EVERY = 256 * 1024 * 1024


def global_log(message):
    elapsed = time.perf_counter() - GLOBAL_START
    print(f"[TOTAL {elapsed:8.2f}s] {message}", flush=True)


def sha256_with_progress(path, dataset_name):

    path = Path(path)

    total_size = path.stat().st_size

    hasher = hashlib.sha256()

    bytes_read = 0
    next_report = REPORT_EVERY

    file_start = time.perf_counter()

    print("\n" + "=" * 100, flush=True)
    print(
        f"HASHING: {dataset_name.upper()}",
        flush=True
    )
    print(
        f"FILE: {path.name}",
        flush=True
    )
    print(
        f"SIZE: {total_size:,} bytes "
        f"({total_size / (1024**3):.3f} GiB)",
        flush=True
    )
    print("=" * 100, flush=True)

    with path.open("rb") as f:

        while True:

            chunk = f.read(CHUNK_SIZE)

            if not chunk:
                break

            hasher.update(chunk)

            bytes_read += len(chunk)

            if (
                bytes_read >= next_report
                or bytes_read == total_size
            ):

                elapsed = time.perf_counter() - file_start

                percent = (
                    bytes_read / total_size * 100
                    if total_size
                    else 100
                )

                mib_read = bytes_read / (1024**2)
                mib_total = total_size / (1024**2)

                speed = (
                    mib_read / elapsed
                    if elapsed > 0
                    else 0
                )

                print(
                    f"  {mib_read:10.1f} / "
                    f"{mib_total:10.1f} MiB | "
                    f"{percent:6.2f}% | "
                    f"{speed:7.1f} MiB/s | "
                    f"{elapsed:7.1f}s",
                    flush=True
                )

                next_report += REPORT_EVERY

    digest = hasher.hexdigest()

    elapsed = time.perf_counter() - file_start

    print(
        f"\nDONE: {dataset_name.upper()} / {path.name}",
        flush=True
    )

    print(
        f"SHA-256: {digest}",
        flush=True
    )

    print(
        f"Time: {elapsed:.2f} seconds",
        flush=True
    )

    return {
        "dataset": dataset_name,
        "file_name": path.name,
        "path": str(path),
        "size_bytes": total_size,
        "sha256": digest,
        "hash_seconds": elapsed
    }


global_log("START — SHA-256 verification")

checksum_records = []

for dataset_name, paths in DATASET_FILES.items():

    global_log(
        f"Starting dataset: {dataset_name}"
    )

    for path in paths:

        record = sha256_with_progress(
            path=path,
            dataset_name=dataset_name
        )

        checksum_records.append(record)

    global_log(
        f"Finished dataset: {dataset_name}"
    )


global_log(
    f"DONE — hashed "
    f"{len(checksum_records)} canonical source files"
)

[TOTAL     0.00s] START — SHA-256 verification
[TOTAL     0.00s] Starting dataset: yelpchi

HASHING: YELPCHI
FILE: YelpChi.mat
SIZE: 207,676,376 bytes (0.193 GiB)
       198.1 /      198.1 MiB | 100.00% |    82.9 MiB/s |     2.4s

DONE: YELPCHI / YelpChi.mat
SHA-256: fedb35a8fa539b27866244d3515a47a76b20080cdacb33112da3458fd2487b42
Time: 2.39 seconds
[TOTAL     2.40s] Finished dataset: yelpchi
[TOTAL     2.40s] Starting dataset: amazon

HASHING: AMAZON
FILE: Amazon.mat
SIZE: 222,634,416 bytes (0.207 GiB)
       212.3 /      212.3 MiB | 100.00% |   122.1 MiB/s |     1.7s

DONE: AMAZON / Amazon.mat
SHA-256: 4b7e3f9cccc62b736792707393ccd74332a1a0592dba128ac6b2989bf1ee9d63
Time: 1.74 seconds
[TOTAL     4.15s] Finished dataset: amazon
[TOTAL     4.15s] Starting dataset: tfinance

HASHING: TFINANCE
FILE: tfinance
SIZE: 682,904,644 bytes (0.636 GiB)
       256.0 /      651.3 MiB |  39.31% |   123.4 MiB/s |     2.1s
       512.0 /      651.3 MiB |  78.62% |   119.5 MiB/s |     4.3s
       651.3

### Step 4C — Save Canonical Dataset Registry

The verified mounted paths, file sizes and SHA-256 values are now written
to benchmark metadata files.

Outputs:

- `dataset_checksums.csv`
- `dataset_registry.json`

These records will later be referenced by model run manifests.

In [11]:
# ============================================================
# STEP 4C — SAVE DATASET REGISTRY
# ============================================================

from pathlib import Path
import pandas as pd
import json
import time


start = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - start
    print(f"[{elapsed:6.2f}s] {message}", flush=True)


log("START — saving canonical dataset registry")

ROOT = Path(
    "/kaggle/working/comp8851_benchmark"
)

MANIFEST_DIR = ROOT / "dataset_manifests"

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Save detailed checksum table
# ------------------------------------------------------------

log("Creating checksum dataframe")

checksum_df = pd.DataFrame(
    checksum_records
)

checksum_csv = (
    MANIFEST_DIR /
    "dataset_checksums.csv"
)

checksum_df.to_csv(
    checksum_csv,
    index=False
)

log(
    f"Saved checksum CSV: {checksum_csv}"
)

# ------------------------------------------------------------
# 2. Build JSON dataset registry
# ------------------------------------------------------------

log("Building dataset registry JSON")

registry = {}

for dataset_name in DATASET_FILES.keys():

    dataset_records = [
        record
        for record in checksum_records
        if record["dataset"] == dataset_name
    ]

    registry[dataset_name] = {
        "verification_status": "SOURCE_FILE_HASHED",
        "files": dataset_records
    }

registry_path = (
    MANIFEST_DIR /
    "dataset_registry.json"
)

with registry_path.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        registry,
        f,
        indent=2
    )

log(
    f"Saved registry JSON: {registry_path}"
)

# ------------------------------------------------------------
# 3. Display final table
# ------------------------------------------------------------

print("\n===== CANONICAL DATASET CHECKSUM TABLE =====", flush=True)

display(
    checksum_df[
        [
            "dataset",
            "file_name",
            "size_bytes",
            "sha256",
            "hash_seconds"
        ]
    ]
)

print("\n===== SAVED FILES =====", flush=True)

print(checksum_csv, flush=True)
print(registry_path, flush=True)

log("DONE — canonical registry saved")

[  0.00s] START — saving canonical dataset registry
[  0.00s] Creating checksum dataframe
[  0.04s] Saved checksum CSV: /kaggle/working/comp8851_benchmark/dataset_manifests/dataset_checksums.csv
[  0.04s] Building dataset registry JSON
[  0.04s] Saved registry JSON: /kaggle/working/comp8851_benchmark/dataset_manifests/dataset_registry.json

===== CANONICAL DATASET CHECKSUM TABLE =====


,dataset,file_name,size_bytes,sha256,hash_seconds
0,yelpchi,YelpChi.mat,207676376,fedb35a8fa539b27866244d3515a47a76b20080cdacb33...,2.391359
1,amazon,Amazon.mat,222634416,4b7e3f9cccc62b736792707393ccd74332a1a0592dba12...,1.741016
2,tfinance,tfinance,682904644,b7d853ec4079e9f7137c03f78a044b1c33d0ff5ed6caa2...,5.546476
3,tsocial,tsocial,4110300257,8d577114cff12f7de35eda2974f17825ef9febe20c661e...,33.870095
4,elliptic,elliptic_txs_features.csv,689683771,fd7f83573443c9e302e371d3f110e3b6224160f5d1ed8a...,6.038702
5,elliptic,elliptic_txs_classes.csv,3305144,93e2e7b2405c735ba752bf6ba06b947561deddd1f5a8fc...,0.040606
6,elliptic,elliptic_txs_edgelist.csv,4470584,a35053ba68a98e4382cae2ba65b9d9e36b23b6439e02df...,0.049291
7,fdcompcn,comp.dgl,1859985,e252b9a6b619b28a7bf6d9f5b16aacc232d43207d87ac2...,0.027616



===== SAVED FILES =====
/kaggle/working/comp8851_benchmark/dataset_manifests/dataset_checksums.csv
/kaggle/working/comp8851_benchmark/dataset_manifests/dataset_registry.json
[  0.09s] DONE — canonical registry saved


### Step 4D — Full Checksum Verification

The table display shortened long SHA-256 values.

This cell prints every checksum in full so the benchmark registry can be
compared against previously audited dataset versions.

A byte-level mismatch does not automatically imply different graph content.
If two serialized graph files differ, their graph structure and labels must
be compared before deciding whether they represent different dataset versions.

In [12]:
# ============================================================
# STEP 4D — PRINT FULL SHA-256 VALUES
# ============================================================

import time

start = time.perf_counter()

def log(message):
    elapsed = time.perf_counter() - start
    print(f"[{elapsed:6.2f}s] {message}", flush=True)

log("START — printing full checksums")

print("\n===== FULL CANONICAL SHA-256 VALUES =====\n", flush=True)

for record in checksum_records:

    print(
        f"Dataset : {record['dataset']}\n"
        f"File    : {record['file_name']}\n"
        f"Size    : {record['size_bytes']:,} bytes\n"
        f"SHA-256 : {record['sha256']}\n",
        flush=True
    )

log("DONE — full checksum output")

[  0.00s] START — printing full checksums

===== FULL CANONICAL SHA-256 VALUES =====

Dataset : yelpchi
File    : YelpChi.mat
Size    : 207,676,376 bytes
SHA-256 : fedb35a8fa539b27866244d3515a47a76b20080cdacb33112da3458fd2487b42

Dataset : amazon
File    : Amazon.mat
Size    : 222,634,416 bytes
SHA-256 : 4b7e3f9cccc62b736792707393ccd74332a1a0592dba128ac6b2989bf1ee9d63

Dataset : tfinance
File    : tfinance
Size    : 682,904,644 bytes
SHA-256 : b7d853ec4079e9f7137c03f78a044b1c33d0ff5ed6caa297b895542a7c8e3700

Dataset : tsocial
File    : tsocial
Size    : 4,110,300,257 bytes
SHA-256 : 8d577114cff12f7de35eda2974f17825ef9febe20c661e95e8d21072a6dfc8d0

Dataset : elliptic
File    : elliptic_txs_features.csv
Size    : 689,683,771 bytes
SHA-256 : fd7f83573443c9e302e371d3f110e3b6224160f5d1ed8a287757936127800ff0

Dataset : elliptic
File    : elliptic_txs_classes.csv
Size    : 3,305,144 bytes
SHA-256 : 93e2e7b2405c735ba752bf6ba06b947561deddd1f5a8fc91e46f6a4c0e439493

Dataset : elliptic
File    : 

## Step 5 — Structural Dataset Verification

Checksum verification confirms file identity, but it does not prove that each
file contains the expected benchmark graph.

This step verifies the actual structure of the mounted datasets before any
model-specific preprocessing begins.

Checks include:

- node and feature dimensions
- label dimensions
- relation matrices
- graph representation
- temporal structure where applicable

YelpChi and Amazon are inspected as MATLAB files.
Elliptic is inspected from its three CSV files.

The serialized DGL datasets (T-Finance, T-Social and FDCompCN) will be checked
separately in a DGL-compatible environment.

### Step 5A — YelpChi Structure

Expected benchmark structure:

- 45,954 review nodes
- 32 features
- one label vector
- three relations:
  - R-U-R
  - R-T-R
  - R-S-R

The file is inspected read-only.

In [13]:
# ============================================================
# STEP 5A — YELPCHI STRUCTURAL VERIFICATION
# ============================================================

from scipy.io import whosmat, loadmat
import time

YELP_PATH = DATASET_FILES["yelpchi"][0]

start = time.perf_counter()

def log(msg):
    print(
        f"[{time.perf_counter() - start:7.2f}s] {msg}",
        flush=True
    )

log("START — YelpChi structural verification")
log(f"File: {YELP_PATH}")

# ------------------------------------------------------------
# 1. Inspect metadata first
# ------------------------------------------------------------

log("Reading MATLAB variable metadata...")

meta = whosmat(YELP_PATH)

print("\n===== YELPCHI MATLAB VARIABLES =====", flush=True)

for name, shape, dtype in meta:
    print(
        f"{name:<20} shape={str(shape):<18} type={dtype}",
        flush=True
    )

log("Metadata scan finished")

# ------------------------------------------------------------
# 2. Load only required variables one at a time
# ------------------------------------------------------------

expected_vars = [
    "features",
    "label",
    "net_rur",
    "net_rtr",
    "net_rsr",
    "homo"
]

results = {}

for var in expected_vars:

    log(f"Loading variable: {var}")

    try:
        obj = loadmat(
            YELP_PATH,
            variable_names=[var]
        )

        if var in obj:
            value = obj[var]
            results[var] = value

            print(
                f"  PASS | {var:<10} | "
                f"shape={value.shape} | "
                f"dtype={value.dtype}",
                flush=True
            )
        else:
            print(
                f"  NOT PRESENT | {var}",
                flush=True
            )

    except Exception as e:
        print(
            f"  ERROR | {var} | "
            f"{type(e).__name__}: {e}",
            flush=True
        )

# ------------------------------------------------------------
# 3. Core summary
# ------------------------------------------------------------

print("\n===== YELPCHI CORE SUMMARY =====", flush=True)

if "features" in results:
    print("Features:", results["features"].shape, flush=True)

if "label" in results:
    print("Labels:", results["label"].shape, flush=True)

for rel in ["net_rur", "net_rtr", "net_rsr"]:
    if rel in results:
        print(
            f"{rel}: {results[rel].shape}",
            flush=True
        )

log("DONE — YelpChi structural verification")

[   0.00s] START — YelpChi structural verification
[   0.00s] File: /kaggle/input/datasets/pathikahmed0007/yelp-chi/YelpChi.mat
[   0.00s] Reading MATLAB variable metadata...

===== YELPCHI MATLAB VARIABLES =====
homo                 shape=(45954, 45954)     type=sparse
net_rur              shape=(45954, 45954)     type=sparse
net_rtr              shape=(45954, 45954)     type=sparse
net_rsr              shape=(45954, 45954)     type=sparse
features             shape=(45954, 32)        type=sparse
label                shape=(1, 45954)         type=int64
[   0.04s] Metadata scan finished
[   0.04s] Loading variable: features
  PASS | features   | shape=(45954, 32) | dtype=float64
[   0.09s] Loading variable: label
  PASS | label      | shape=(1, 45954) | dtype=int64
[   0.09s] Loading variable: net_rur
  PASS | net_rur    | shape=(45954, 45954) | dtype=float64
[   0.10s] Loading variable: net_rtr
  PASS | net_rtr    | shape=(45954, 45954) | dtype=float64
[   0.13s] Loading variable: net

### Step 5B — Amazon Structure

Expected graph structure:

- 11,944 total graph nodes
- 25 features
- one user node type
- three relations:
  - U-P-U
  - U-S-U
  - U-V-U

The file is inspected read-only.

In [14]:
# ============================================================
# STEP 5B — AMAZON STRUCTURAL VERIFICATION
# ============================================================

from scipy.io import whosmat, loadmat
import time

AMAZON_PATH = DATASET_FILES["amazon"][0]

start = time.perf_counter()

def log(msg):
    print(
        f"[{time.perf_counter() - start:7.2f}s] {msg}",
        flush=True
    )

log("START — Amazon structural verification")
log(f"File: {AMAZON_PATH}")

# ------------------------------------------------------------
# 1. Metadata
# ------------------------------------------------------------

log("Reading MATLAB variable metadata...")

meta = whosmat(AMAZON_PATH)

print("\n===== AMAZON MATLAB VARIABLES =====", flush=True)

for name, shape, dtype in meta:
    print(
        f"{name:<20} shape={str(shape):<18} type={dtype}",
        flush=True
    )

log("Metadata scan finished")

# ------------------------------------------------------------
# 2. Required variables
# ------------------------------------------------------------

expected_vars = [
    "features",
    "label",
    "net_upu",
    "net_usu",
    "net_uvu",
    "homo"
]

results = {}

for var in expected_vars:

    log(f"Loading variable: {var}")

    try:
        obj = loadmat(
            AMAZON_PATH,
            variable_names=[var]
        )

        if var in obj:
            value = obj[var]
            results[var] = value

            print(
                f"  PASS | {var:<10} | "
                f"shape={value.shape} | "
                f"dtype={value.dtype}",
                flush=True
            )
        else:
            print(
                f"  NOT PRESENT | {var}",
                flush=True
            )

    except Exception as e:
        print(
            f"  ERROR | {var} | "
            f"{type(e).__name__}: {e}",
            flush=True
        )

# ------------------------------------------------------------
# 3. Summary
# ------------------------------------------------------------

print("\n===== AMAZON CORE SUMMARY =====", flush=True)

if "features" in results:
    print("Features:", results["features"].shape, flush=True)

if "label" in results:
    print("Labels:", results["label"].shape, flush=True)

for rel in ["net_upu", "net_usu", "net_uvu"]:
    if rel in results:
        print(
            f"{rel}: {results[rel].shape}",
            flush=True
        )

log("DONE — Amazon structural verification")

[   0.00s] START — Amazon structural verification
[   0.00s] File: /kaggle/input/datasets/pathikahmed0007/amazon/Amazon.mat
[   0.00s] Reading MATLAB variable metadata...

===== AMAZON MATLAB VARIABLES =====
homo                 shape=(11944, 11944)     type=sparse
net_upu              shape=(11944, 11944)     type=sparse
net_usu              shape=(11944, 11944)     type=sparse
net_uvu              shape=(11944, 11944)     type=sparse
features             shape=(11944, 25)        type=sparse
label                shape=(1, 11944)         type=double
[   0.02s] Metadata scan finished
[   0.02s] Loading variable: features
  PASS | features   | shape=(11944, 25) | dtype=float64
[   0.03s] Loading variable: label
  PASS | label      | shape=(1, 11944) | dtype=float64
[   0.03s] Loading variable: net_upu
  PASS | net_upu    | shape=(11944, 11944) | dtype=float64
[   0.04s] Loading variable: net_usu
  PASS | net_usu    | shape=(11944, 11944) | dtype=float64
[   0.19s] Loading variable: net_u

### Step 5C — Elliptic Structure

Elliptic is represented by three CSV files:

- transaction features
- transaction classes
- directed transaction edges

This step verifies:

- number of transactions
- CSV feature dimension
- modelling feature dimension
- time-step range
- label counts
- directed edge count

The large feature file is read in chunks so progress remains visible.

In [15]:
# ============================================================
# STEP 5C — ELLIPTIC STRUCTURAL VERIFICATION
# ============================================================

import pandas as pd
import time

start = time.perf_counter()

def log(msg):
    print(
        f"[{time.perf_counter() - start:7.2f}s] {msg}",
        flush=True
    )

ELLIPTIC_PATHS = {
    p.name: p
    for p in DATASET_FILES["elliptic"]
}

FEATURES_PATH = ELLIPTIC_PATHS[
    "elliptic_txs_features.csv"
]

CLASSES_PATH = ELLIPTIC_PATHS[
    "elliptic_txs_classes.csv"
]

EDGES_PATH = ELLIPTIC_PATHS[
    "elliptic_txs_edgelist.csv"
]

log("START — Elliptic structural verification")

# ------------------------------------------------------------
# 1. Features
# ------------------------------------------------------------

log("Scanning feature CSV in 25,000-row chunks")

feature_rows = 0
feature_cols = None
time_min = None
time_max = None

for chunk_no, chunk in enumerate(
    pd.read_csv(
        FEATURES_PATH,
        header=None,
        chunksize=25_000
    ),
    start=1
):

    feature_rows += len(chunk)

    if feature_cols is None:
        feature_cols = chunk.shape[1]

    local_min = chunk.iloc[:, 1].min()
    local_max = chunk.iloc[:, 1].max()

    time_min = (
        local_min if time_min is None
        else min(time_min, local_min)
    )

    time_max = (
        local_max if time_max is None
        else max(time_max, local_max)
    )

    print(
        f"  FEATURES chunk {chunk_no:>2} | "
        f"{feature_rows:>7,} rows processed",
        flush=True
    )

log("Feature CSV scan complete")

# ------------------------------------------------------------
# 2. Classes
# ------------------------------------------------------------

log("Loading class CSV")

classes = pd.read_csv(CLASSES_PATH)

log(
    f"Class CSV loaded: {len(classes):,} rows"
)

# ------------------------------------------------------------
# 3. Edges
# ------------------------------------------------------------

log("Scanning edge CSV in 100,000-row chunks")

edge_rows = 0

for chunk_no, chunk in enumerate(
    pd.read_csv(
        EDGES_PATH,
        chunksize=100_000
    ),
    start=1
):

    edge_rows += len(chunk)

    print(
        f"  EDGES chunk {chunk_no:>2} | "
        f"{edge_rows:>7,} rows processed",
        flush=True
    )

log("Edge CSV scan complete")

# ------------------------------------------------------------
# 4. Labels
# ------------------------------------------------------------

label_col = classes.columns[-1]

label_counts = (
    classes[label_col]
    .astype(str)
    .value_counts(dropna=False)
)

# ------------------------------------------------------------
# 5. Final summary
# ------------------------------------------------------------

print("\n===== ELLIPTIC CORE SUMMARY =====", flush=True)

print(
    "Transaction rows :",
    f"{feature_rows:,}",
    flush=True
)

print(
    "CSV columns      :",
    feature_cols,
    flush=True
)

if feature_cols is not None:
    print(
        "Model features   :",
        feature_cols - 2,
        "(excluding txId + time_step)",
        flush=True
    )

print(
    "Time-step range  :",
    f"{time_min} to {time_max}",
    flush=True
)

print(
    "Class rows       :",
    f"{len(classes):,}",
    flush=True
)

print(
    "Directed edges   :",
    f"{edge_rows:,}",
    flush=True
)

print("\n===== LABEL COUNTS =====", flush=True)
print(label_counts, flush=True)

log("DONE — Elliptic structural verification")

[   0.00s] START — Elliptic structural verification
[   0.00s] Scanning feature CSV in 25,000-row chunks
  FEATURES chunk  1 |  25,000 rows processed
  FEATURES chunk  2 |  50,000 rows processed
  FEATURES chunk  3 |  75,000 rows processed
  FEATURES chunk  4 | 100,000 rows processed
  FEATURES chunk  5 | 125,000 rows processed
  FEATURES chunk  6 | 150,000 rows processed
  FEATURES chunk  7 | 175,000 rows processed
  FEATURES chunk  8 | 200,000 rows processed
  FEATURES chunk  9 | 203,769 rows processed
[   9.17s] Feature CSV scan complete
[   9.17s] Loading class CSV
[   9.22s] Class CSV loaded: 203,769 rows
[   9.22s] Scanning edge CSV in 100,000-row chunks
  EDGES chunk  1 | 100,000 rows processed
  EDGES chunk  2 | 200,000 rows processed
  EDGES chunk  3 | 234,355 rows processed
[   9.28s] Edge CSV scan complete

===== ELLIPTIC CORE SUMMARY =====
Transaction rows : 203,769
CSV columns      : 167
Model features   : 165 (excluding txId + time_step)
Time-step range  : 1 to 49
Class r

## Dataset Registry — Current Verification Status

Canonical source files for all six benchmark datasets have been mounted,
identified and SHA-256 hashed.

Structural verification completed:

- YelpChi — PASS
- Amazon — PASS
- Elliptic — PASS

Serialized DGL datasets:

- T-Finance — source registered; structural verification pending in the
  BWGNN-compatible environment.
- T-Social — source hash matches the previously audited official graph;
  deserialization confirmation pending in the BWGNN-compatible environment.
- FDCompCN — source registered; DGL structural verification pending in a
  compatible environment.

The shared registry notebook does not install a model-specific DGL stack.

In [ ]:
# ============================================================
# STEP 5D — UPDATE DATASET REGISTRY STATUS
# ============================================================

import json
import time
from pathlib import Path

start = time.perf_counter()

def log(msg):
    print(
        f"[{time.perf_counter() - start:6.2f}s] {msg}",
        flush=True
    )

log("START — updating verification status")

registry_path = Path(
    "/kaggle/working/comp8851_benchmark/"
    "dataset_manifests/dataset_registry.json"
)

with registry_path.open("r", encoding="utf-8") as f:
    registry = json.load(f)

registry["yelpchi"]["structural_status"] = "PASS"
registry["amazon"]["structural_status"] = "PASS"
registry["elliptic"]["structural_status"] = "PASS"

registry["tfinance"]["structural_status"] = (
    "PENDING_DGL_VERIFICATION"
)

registry["tsocial"]["structural_status"] = (
    "HASH_MATCH_PASS_DGL_LOAD_PENDING"
)

registry["fdcompcn"]["structural_status"] = (
    "PENDING_DGL_VERIFICATION"
)

with registry_path.open("w", encoding="utf-8") as f:
    json.dump(registry, f, indent=2)

print("\n===== CURRENT DATASET GATE STATUS =====", flush=True)

for name, info in registry.items():
    print(
        f"{name:<12} : "
        f"{info.get('structural_status', 'NOT SET')}",
        flush=True
    )

log(f"Saved: {registry_path}")
log("DONE")